In [1]:
import pandas as pd 
import os
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import mne
import numpy as np
from tqdm import tqdm
import re
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from multiprocessing import Pool
import warnings

warnings.filterwarnings("ignore")

os.chdir('../..')

# Load Morlet PSDs

In [2]:
#loaded = np.load('./Generated/Spectrums/psds_array_morlet.npz')
loaded = np.load('C:/Users/dasga/PycharmProjects/EEG/EEG-Image-Reconstruction/Generated/Spectrums/psds_array_morlet.npz')

results_arr = []

i = 0
while f'psd_{i}' in loaded:
    psd = loaded[f'psd_{i}']
    s_id = int(loaded[f'subject_id_{i}'])
    t_id = int(loaded[f'trial_id_{i}'])
    gender = str(loaded[f'gender_{i}'])
    handiness = str(loaded[f'handiness_{i}'])
    age = int(loaded[f'age_{i}'])
    
    results_arr.append([psd, s_id, t_id, gender, handiness, age])
    i += 1

psd, s_id, t_id, gender, handiness, age = results_arr[0]
psd.shape

(63, 80)

In [3]:
# Parse into two lists will be used furthere
psds_array = [item[0] for item in results_arr]
psds_array = np.array(psds_array)                            # Convert list of PSDs to a numpy array
psds_array = psds_array.reshape((psds_array.shape[0], -1))   # Vectorize 

metadata = [item[1:] for item in results_arr]

psds_array.shape

(31, 5040)

### Normalize Each Spectrum

# Load MetaData table

In [4]:
file_path = r'C:/Users/dasga/PycharmProjects/EEG/EEG-Image-Reconstruction/Supplementary/Experiment_Metadata.xlsx'
meta = pd.read_excel(file_path, header=1).rename(columns={
    'Subject ID'          : 'Subject_id',
    'Время начала записи' : 'Time',
})

meta['Subject_id'] = (
    meta['Subject_id']
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype('float')
)
meta = meta[meta['Subject_id'].notna()].copy()
meta['Subject_id'] = meta['Subject_id'].astype(int)

s  = meta['Time'].astype(str).str.strip()                    
n  = pd.to_numeric(s, errors='coerce')                        
dt_str = pd.to_datetime(s, errors='coerce', dayfirst=True, infer_datetime_format=True)
dt_num = pd.to_datetime(n, errors='coerce', origin='1899-12-30', unit='D')
dt = dt_str.fillna(dt_num)                                   

meta['Hour'] = dt.dt.hour
meta['Time'] = dt.dt.strftime('%H:%M')

meta_idx = (meta[['Subject_id', 'Hour', 'Time']]
            .dropna(subset=['Hour'])                         
            .drop_duplicates('Subject_id', keep='last'))

m = np.array(metadata, dtype=object)  # [s_id, t_id, gender, handiness, age]
df_out = pd.DataFrame({'Subject_ID': m[:,0].astype(int), 'Trial_ID': m[:,1].astype(int)})
df_out = df_out.merge(meta_idx, left_on='Subject_ID', right_on='Subject_id', how='left').drop(columns=['Subject_id'])

df_out['Condition'] = pd.cut(
    df_out['Hour'],
    bins=[-0.1, 10, 18, 24],
    labels=['Other', 'Day', 'Evening'],
    right=False,
    include_lowest=True
).astype(object).fillna('Other')

df_out = df_out[['Subject_ID', 'Trial_ID', 'Time', 'Condition']]

print(df_out['Condition'].value_counts(dropna=False))
df_out.head(10)


Condition
Day        17
Evening    14
Name: count, dtype: int64


,Subject_ID,Trial_ID,Time,Condition
0,4,2,16:00,Day
1,4,1,16:00,Day
2,14,1,13:00,Day
3,15,2,16:00,Day
4,15,1,16:00,Day
5,1,2,18:00,Evening
6,1,1,18:00,Evening
7,5,2,19:00,Evening
8,5,1,19:00,Evening
9,3,2,13:00,Day


In [6]:
def map_cond(c):
    if pd.isna(c): return None
    if c == 'Day': return 'Day'
    if c == 'Evening': return 'Night'
    if c == 'Night': return 'Night'
    return None  

mapped = df_out['Condition'].apply(map_cond)

row_choice = pd.Series(mapped).where(pd.Series(mapped).notna())
mode_global = pd.Series([x for x in row_choice if pd.notna(x)]).mode().iloc[0] if pd.Series(row_choice).notna().any() else 'Day'

subj_choice = (df_out
    .assign(_choice=row_choice)
    .sort_values(['Subject_ID','Trial_ID'])
    .groupby('Subject_ID', as_index=False)['_choice']
    .agg(lambda s: next((x for x in s if pd.notna(x)), mode_global))
    .rename(columns={'_choice':'Cond_final'})
)
subj2cond = dict(zip(subj_choice['Subject_ID'], subj_choice['Cond_final']))
df_out['Cond_final'] = df_out['Subject_ID'].map(subj2cond)

print('Counts by subject condition:', pd.Series(subj2cond).value_counts().to_dict())
print(df_out['Cond_final'].value_counts(dropna=False))


first_psd = results_arr[0][0]
n_channels, n_freqs = first_psd.shape

day_vectors   = {ch: [] for ch in range(n_channels)}
night_vectors = {ch: [] for ch in range(n_channels)}
placed = {'Day': 0, 'Night': 0}

for psd, s_id, t_id, gender, handiness, age in results_arr:
    cond = subj2cond.get(int(s_id), mode_global)
    if cond == 'Day':
        for ch in range(n_channels):
            day_vectors[ch].append(psd[ch, :].astype(float))
        placed['Day'] += 1
    else:
        for ch in range(n_channels):
            night_vectors[ch].append(psd[ch, :].astype(float))
        placed['Night'] += 1

print('Placed trials:', placed)

day_matrices   = {ch: (np.vstack(v) if v else np.empty((0, n_freqs))) for ch, v in day_vectors.items()}
night_matrices = {ch: (np.vstack(v) if v else np.empty((0, n_freqs))) for ch, v in night_vectors.items()}

for ch in range(min(5, n_channels)):
    print(f'ch {ch}: day_matrix={day_matrices[ch].shape}, night_matrix={night_matrices[ch].shape}')


Counts by subject condition: {'Day': 9, 'Night': 7}
Cond_final
Day      17
Night    14
Name: count, dtype: int64
Placed trials: {'Day': 17, 'Night': 14}
ch 0: day_matrix=(17, 80), night_matrix=(14, 80)
ch 1: day_matrix=(17, 80), night_matrix=(14, 80)
ch 2: day_matrix=(17, 80), night_matrix=(14, 80)
ch 3: day_matrix=(17, 80), night_matrix=(14, 80)
ch 4: day_matrix=(17, 80), night_matrix=(14, 80)


In [8]:
# Welch t-test 
from scipy import stats
from statsmodels.stats.power import TTestIndPower

n_freqs = day_matrices[0].shape[1]  # количество частот из первого канала
freqs = np.arange(n_freqs, dtype=float)


power_calc = TTestIndPower()
alpha = 0.05

def effect_size_d(x, y):
    # Cohen's d: разница средних / std
    nx, ny = len(x), len(y)
    sp = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / sp


rows = []
for ch, D in day_matrices.items():
    N = night_matrices.get(ch, None)

    n_freqs = D.shape[1]
    for fi in range(n_freqs):
        x = D[:, fi]
        y = N[:, fi]

        # Welch t-test
        t, p = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')

        # Power — через effect size
        d = effect_size_d(x, y)
        n1, n2 = len(x), len(y)
        ratio = n2 / n1 
        power = power_calc.solve_power(effect_size=np.abs(d), nobs1=n1, ratio=ratio,
                                           alpha=alpha, alternative='two-sided')
        i = True

        rows.append({
            "channel": ch,
            "freq_hz": float(freqs[fi]),
            "n_day": len(x),
            "n_night": len(y),
            "t_stat": float(t),
            "p_value": float(p),
            "power": float(power),
            "Hyp": bool(i) if power > p else not i
        })

welch_df = pd.DataFrame(rows).sort_values(["p_value", "channel", "freq_hz"])
print(welch_df.head(15))


      channel  freq_hz  n_day  n_night    t_stat   p_value     power   Hyp
1248       15     48.0     17       14  3.813698  0.001155  0.923115  True
1249       15     49.0     17       14  3.811613  0.001167  0.922694  True
1247       15     47.0     17       14  3.804396  0.001173  0.922097  True
1250       15     50.0     17       14  3.801488  0.001200  0.921250  True
1246       15     46.0     17       14  3.780980  0.001229  0.919248  True
1251       15     51.0     17       14  3.787449  0.001244  0.919282  True
1252       15     52.0     17       14  3.773864  0.001288  0.917340  True
1259       15     59.0     17       14  3.789747  0.001297  0.918178  True
1258       15     58.0     17       14  3.785639  0.001303  0.917751  True
1260       15     60.0     17       14  3.786575  0.001308  0.917707  True
1257       15     57.0     17       14  3.777675  0.001319  0.916864  True
1253       15     53.0     17       14  3.764285  0.001323  0.915896  True
1256       15     56.0   

Ok. Try with bands 

In [24]:
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 45),
}

band_rows = []


band_idx = {b: np.where((freqs >= lo) & (freqs < hi))[0] for b,(lo,hi) in bands.items()}

for ch, D in day_matrices.items():
    N = night_matrices.get(ch, np.empty((0, len(freqs))))
    if D.size == 0 or N.size == 0:
        continue
    for b, idx in band_idx.items():
        if len(idx) == 0:
            continue
        x = np.nanmean(D[:, idx], axis=1)  # средняя мощность в диапазоне по записи
        y = np.nanmean(N[:, idx], axis=1)

        t, p = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
        d = effect_size_d(x, y)
        try:
            n1, n2 = len(x), len(y)
            ratio = n2 / n1 if n1 > 0 else 1.0
            power = power_calc.solve_power(effect_size=np.abs(d), nobs1=n1, ratio=ratio,
                                           alpha=alpha, alternative='two-sided')
        except Exception:
            power = np.nan
        i = True
        band_rows.append({
            "channel": ch,
            "band": b,
            "n_day": len(x),
            "n_night": len(y),
            "t_stat": float(t),
            "p_value": float(p),
            "power": float(power) if np.isfinite(power) else np.nan,
            "mean_day": float(np.nanmean(x)),
            "mean_night": float(np.nanmean(y)),
            "HYP": bool(i) if power > p and p <= 0.01 else not i
        })

band_df = pd.DataFrame(band_rows).sort_values(["p_value", "power"])
print(band_df.head(15))

     channel   band  n_day  n_night    t_stat   p_value     power  mean_day  \
309       61  gamma     17       14  2.982359  0.007244  0.759382  0.068095   
79        15  gamma     17       14  2.852517  0.008479  0.742683  0.085418   
144       28  gamma     17       14  2.882454  0.009885  0.721337  0.110744   
44         8  gamma     17       14  2.819965  0.010853  0.707212  0.187004   
199       39  gamma     17       14  2.738017  0.011431  0.701497  0.076319   
9          1  gamma     17       14  2.733882  0.013751  0.674356  0.074757   
304       60  gamma     17       14  2.621827  0.017595  0.636724  0.115811   
159       31  gamma     17       14  2.502028  0.022307  0.598172  0.226422   
104       20  gamma     17       14  2.395022  0.024183  0.593344  0.109228   
14         2  gamma     17       14  2.366342  0.029496  0.551268  0.149265   
302       60  alpha     17       14  2.275286  0.030465  0.575905  0.160375   
54        10  gamma     17       14  2.202047  0.037